[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/timeseries/04-classical-models.ipynb)

# Classical Models: ETS, ARIMA and Regression with ARMA Errors

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Fit exponential-smoothing and (S)ARIMA models and read what the fitted parameters mean.</li>
<li>Explain why a model without a seasonal component reduces to the naive forecast.</li>
<li>Handle a long seasonal period with regression terms plus ARMA errors, and check convergence.</li>
<li>Judge a classical model against a strong baseline without over-reading a small edge.</li>
</ul>
</div>

**Prerequisites:** Pages 1 to 3 of this track. **Dataset:** NYC Taxi (Jan to Mar 2023). **Time:** about 40 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import forecasting as fc, style
from mlp_data.timeseries import taxi_hourly

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

## The situation

Baselines are set: a four-week same-hour average gets about 280 pickups an hour of error. Now we try the **classical statistical models** that dominated forecasting before machine learning, and which are still the right tool for many series: exponential smoothing (ETS) and ARIMA with its seasonal and regression extensions.

The aim is not to crown a winner but to learn what each family can and cannot do, and to see how much of the work is choosing a *form* that matches the seasonality.

In [2]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning

y = taxi_hourly(["2023-01", "2023-02", "2023-03"]).asfreq("h").interpolate()

## 1. The models on a simpler series: daily totals

Hourly data with a 168-hour season is awkward for these models (we return to that in section 2). So first learn them on **daily totals**, where the weekly season is just 7 steps. The task: at each midnight, forecast the next 7 days. We test on 21 origins from 4 to 24 March (a stretch with no public holidays), and compare with the same baselines as before.

<div class="admonition math">
<p class="admonition-title">Maths ↔ code</p>
<p><b>Exponential smoothing</b> keeps a running <em>level</em> (and optionally a <em>trend</em> and a <em>seasonal</em> pattern) and updates each with a smoothing weight: the level is $\ell_t = \alpha\,(y_t - s_{t-m}) + (1-\alpha)\,\ell_{t-1}$ in the additive-seasonal form. A weight near 1 means "trust the latest value", near 0 means "trust the history". In code: <code>ExponentialSmoothing(y, trend=, seasonal=, seasonal_periods=)</code>. The naming ETS(Error, Trend, Seasonal) uses A (additive), M (multiplicative) or N (none).</p>
<p><b>ARIMA(p, d, q)</b> differences the series <em>d</em> times, then models what is left as <em>p</em> autoregressive terms plus <em>q</em> moving-average terms of past errors. <b>SARIMA</b> adds the same ideas at the seasonal lag: <code>seasonal_order=(P, D, Q, m)</code>. <b>SARIMAX</b> also allows regressors <em>X</em>, with ARMA-modelled errors.</p>
</div>

In [3]:
daily = y.resample("D").sum()
daily.index.freq = "D"
H = 7
origins = fc.rolling_origins(daily.index, first_origin="2023-03-04", last_origin="2023-03-24")
train_d = daily[:"2023-03-03"]
print(len(origins), "origins x", H, "days =", len(origins) * H, "forecast values")

def ets(**kw):
    return lambda hist, idx: ExponentialSmoothing(hist, initialization_method="estimated", **kw).fit().forecast(len(idx)).to_numpy()

def arima(order, seasonal_order=(0, 0, 0, 0)):
    return lambda hist, idx: SARIMAX(hist, order=order, seasonal_order=seasonal_order).fit(disp=False).forecast(len(idx)).to_numpy()

models = {
    "naive (last value)": lambda hist, idx: fc.seasonal_naive(hist, idx, 1),
    "seasonal naive (same weekday last week)": lambda hist, idx: fc.seasonal_naive(hist, idx, 7),
    "ETS(A,N,N): level only": ets(),
    "ETS(A,Ad,N): damped trend, no season": ets(trend="add", damped_trend=True),
    "ETS(A,N,A): level + weekly season": ets(seasonal="add", seasonal_periods=7),
    "ETS(A,N,M): multiplicative weekly season": ets(seasonal="mul", seasonal_periods=7),
    "ARIMA(1,1,1): no season": arima((1, 1, 1)),
    "SARIMA(1,0,1)(0,1,1,7)": arima((1, 0, 1), (0, 1, 1, 7)),
}
rows, runs_d = {}, {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                                # reviewed below: convergence is checked for the hourly models
    for name, f in models.items():
        runs_d[name] = fc.backtest(daily, f, origins=origins, horizon=H)
        rows[name] = fc.score(runs_d[name], train_d, m=7)
table = pd.DataFrame(rows).T
table.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "sMAPE": "{:.1%}", "MASE": "{:.2f}"})

21 origins x 7 days = 147 forecast values


,MAE,RMSE,sMAPE,MASE
naive (last value),"11,267","13,696",10.4%,2.11
seasonal naive (same weekday last week),"5,597","6,698",5.1%,1.05
"ETS(A,N,N): level only","11,267","13,696",10.4%,2.11
"ETS(A,Ad,N): damped trend, no season","11,267","13,696",10.4%,2.11
"ETS(A,N,A): level + weekly season","4,585","5,494",4.2%,0.86
"ETS(A,N,M): multiplicative weekly season","4,824","5,845",4.4%,0.91
"ARIMA(1,1,1): no season","10,531","12,833",9.7%,1.98
"SARIMA(1,0,1)(0,1,1,7)","5,369","6,489",4.9%,1.01


Read the table in groups.

- **Models with no seasonal part are stuck at the naive forecast's level (MASE 2.0 to 2.1), or barely better.** They have no way to represent the weekly cycle, so the best they can do is a flat line. The level-only smoother does exactly what the naive forecast does; look at why:

In [4]:
fit = ExponentialSmoothing(train_d, initialization_method="estimated").fit()
print("smoothing weight alpha chosen for the level-only model:", round(fit.params["smoothing_level"], 3))
print("MAE: level-only ETS =", round(table.loc["ETS(A,N,N): level only", "MAE"]), "| naive =", round(table.loc["naive (last value)", "MAE"]))

smoothing weight alpha chosen for the level-only model: 1.0
MAE: level-only ETS = 11267 | naive = 11267


The fitted weight is 1.0: "trust only the latest value", which *is* the naive forecast. The data told the optimiser that, without a seasonal term, nothing beats the last value.

- **Adding the weekly season is what matters.** ETS(A,N,A) cuts the error by about **18% relative to seasonal naive** (MASE 0.86 against 1.05). The weekly pattern is learned from all the weeks with smoothing, rather than copied from one week.
- **Multiplicative seasonality is a little worse than additive** here, consistent with the mild scaling we saw earlier, and not worth the extra complexity.
- **Ordinary ARIMA(1,1,1) gains almost nothing** (MASE 1.98 against 2.11). **Seasonal ARIMA is on par with the seasonal naive forecast** (MASE about 1.0): it also learns the weekly season, but its estimates from only 62 training days are noisy.

How would you check whether a fitted SARIMA has captured the structure? Look at the **residuals**. If the model is adequate they should look like noise; the Ljung-Box test checks that formally (null hypothesis: no autocorrelation up to the lag tested).

In [5]:
sarima = SARIMAX(train_d, order=(1, 0, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False)
resid = sarima.resid[7:]                                   # drop the warm-up residuals created by the seasonal differencing
print(sarima.summary().tables[1].as_text()[:520])
acorr_ljungbox(resid, lags=[7, 14])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.8469      0.046     18.451      0.000       0.757       0.937
ma.L1         -0.7850      0.056    -14.033      0.000      -0.895      -0.675
ma.S.L7       -0.0570      0.055     -1.031      0.303      -0.165       0.051
sigma2      3.112e+07   2.22e-10    1.4e+17   


,lb_stat,lb_pvalue
7,27.65,0.00
14,40.33,0.00


The p-values are tiny: the residuals still contain autocorrelation, so this SARIMA has **not** captured everything. (The usual reaction is to try more terms; with 62 observations that quickly overfits, which is what the backtest is for.) Diagnostics suggest where a model is deficient. The **backtest** tells you whether the deficiency matters for forecasting.

## 2. Back to hourly: the long-season problem

On hourly data the weekly season has 168 steps. ETS must estimate a state for each of the 168 positions, and a seasonal ARIMA with $m = 168$ is impractically slow. Two practical routes exist: let ETS keep its 168 seasonal states, or **take the seasonality out of the ARIMA** by handing it to a regression (the "X" in SARIMAX) and keeping ARMA only for the errors.

We go back to the 24-hour-ahead task and the 14 test origins (18 to 31 March) from the baselines page, and keep the best baseline in view.

In [6]:
test_origins = fc.rolling_origins(y.index, first_origin="2023-03-18")
train = y[:"2023-03-17"]
H = 24
four_week = fc.backtest(y, lambda hist, idx: fc.seasonal_mean(hist, idx, weeks=4), origins=test_origins, horizon=H)
print("baseline to beat (4-week average):", {k: round(v, 3) for k, v in fc.score(four_week, train).items()})

def ets_168(weeks):
    return lambda hist, idx: ExponentialSmoothing(hist.iloc[-168 * weeks:], seasonal="add", seasonal_periods=168,
                                                   initialization_method="heuristic").fit().forecast(len(idx)).to_numpy()

rows = {}
for weeks in (4, 6, 8):
    bt = fc.backtest(y, ets_168(weeks), origins=test_origins, horizon=H)
    rows[f"ETS(A,N,A), m=168, fitted on the last {weeks} weeks"] = fc.score(bt, train)
pd.DataFrame(rows).T.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "sMAPE": "{:.1%}", "MASE": "{:.2f}"})

baseline to beat (4-week average): {'MAE': 276.629, 'RMSE': 380.879, 'sMAPE': 0.071, 'MASE': 0.801}


,MAE,RMSE,sMAPE,MASE
"ETS(A,N,A), m=168, fitted on the last 4 weeks",331,422,14.4%,0.96
"ETS(A,N,A), m=168, fitted on the last 6 weeks",340,433,12.3%,0.98
"ETS(A,N,A), m=168, fitted on the last 8 weeks",395,488,15.5%,1.14


ETS with 168 seasonal states is **sensitive to how much history you give it**: even at its best (the last 4 weeks, MAE 331) it is about 20% worse than the baseline's 277, and with 8 weeks (395) it is clearly worse. Having more data makes it *worse*, because its level and smoothing weights have to adapt to a drifting series. That sensitivity is a warning sign: a model whose quality changes this much with a setting that has no principled value is hard to trust.

The second route is **harmonic regression**: describe the seasonal shape with a handful of sine and cosine terms (Fourier terms) and let ARIMA model what is left.

In [7]:
def fourier(index, k24, k168):
    t = (index - pd.Timestamp("2023-01-01")).total_seconds().to_numpy() / 3600
    cols = {}
    for period, k in ((24, k24), (168, k168)):
        for j in range(1, k + 1):
            cols[f"sin_{period}_{j}"] = np.sin(2 * np.pi * j * t / period)
            cols[f"cos_{period}_{j}"] = np.cos(2 * np.pi * j * t / period)
    return pd.DataFrame(cols, index=index)

def regression_arma(make_exog, order=(1, 0, 0), weeks=6, method="lbfgs"):
    """Seasonality from a regression on exogenous columns, short-term memory from ARMA errors."""
    counter = {"fits": 0, "not_converged": 0}
    def forecaster(hist, idx):
        hist = hist.iloc[-168 * weeks:]
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = SARIMAX(hist, exog=make_exog(hist.index), order=order, trend="c").fit(disp=False, maxiter=200, method=method)
        counter["fits"] += 1
        counter["not_converged"] += any(issubclass(w.category, ConvergenceWarning) for w in caught)
        return model.forecast(len(idx), exog=make_exog(idx)).to_numpy()
    forecaster.counter = counter
    return forecaster

harmonic = regression_arma(lambda idx: fourier(idx, 10, 10), order=(1, 0, 0), method="powell")
bt_h = fc.backtest(y, harmonic, origins=test_origins, horizon=H)
print("Fourier terms (10 daily + 10 weekly harmonics) + AR(1) errors:", {k: round(v, 3) for k, v in fc.score(bt_h, train).items()})
print("fits that did not converge:", harmonic.counter["not_converged"], "of", harmonic.counter["fits"])

Fourier terms (10 daily + 10 weekly harmonics) + AR(1) errors: {'MAE': 518.497, 'RMSE': 655.478, 'sMAPE': 0.21, 'MASE': 1.501}
fits that did not converge: 0 of 14


The harmonic version is much **worse than the baseline** (MAE about 520 against 277), and this is not an optimiser failure: we used a method (Powell) that converged on all 14 fits. (With the default optimiser, 14 of 14 fits of this model reported *no convergence* in my tests, a reminder to count the warnings rather than silence them.) Twenty smooth waves cannot reproduce the *sharp* edges of the real weekly shape: the sudden morning ramp, the evening peak, the Friday-night surge. Adding harmonics is a bias-variance trade: too few lose the shape, too many chase noise and slow the fit.

When the season is as short as one week, there is an exact alternative: a **separate regression coefficient for each of the 168 weekday-hour slots** (dummy variables), with AR errors on top.

In [8]:
def slot_dummies(index):
    slot = pd.Series(index.dayofweek * 24 + index.hour, index=index).astype("category")
    return pd.get_dummies(slot).astype(float).reindex(columns=range(168), fill_value=0.0)

slots = regression_arma(slot_dummies, order=(1, 0, 0))
bt_s = fc.backtest(y, lambda hist, idx: slots(hist, idx), origins=test_origins, horizon=H)
six_week = fc.backtest(y, lambda hist, idx: fc.seasonal_mean(hist, idx, weeks=6), origins=test_origins, horizon=H)

compare = pd.DataFrame({
    "4-week seasonal average": fc.score(four_week, train),
    "6-week seasonal average": fc.score(six_week, train),
    "168 slot dummies + AR(1) errors (6 weeks)": fc.score(bt_s, train),
}).T
print("fits that did not converge:", slots.counter["not_converged"], "of", slots.counter["fits"])
compare.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "sMAPE": "{:.1%}", "MASE": "{:.2f}"})

fits that did not converge: 0 of 14


,MAE,RMSE,sMAPE,MASE
4-week seasonal average,277,381,7.1%,0.80
6-week seasonal average,262,375,6.7%,0.76
168 slot dummies + AR(1) errors (6 weeks),262,364,7.2%,0.76


This is the first model to **match** the best baseline, and the comparison is instructive. The regression with AR errors scores essentially the same MAE as the **plain six-week seasonal average** (262 against 262), with a slightly lower RMSE. That is no accident: with a dummy for every weekday-hour slot, the regression's fitted values *are* seasonal averages over the training weeks. The only thing ARIMA adds is the AR(1) error term. Where does that help?

In [9]:
by_step = pd.DataFrame({
    "6-week average": six_week.assign(e=(six_week["actual"] - six_week["forecast"]).abs()).groupby("step")["e"].mean(),
    "slot dummies + AR(1)": bt_s.assign(e=(bt_s["actual"] - bt_s["forecast"]).abs()).groupby("step")["e"].mean(),
})
by_step["gain"] = by_step["6-week average"] - by_step["slot dummies + AR(1)"]
for label, steps in [("steps 1-2", slice(1, 2)), ("steps 3-6", slice(3, 6)), ("steps 7-24", slice(7, 24))]:
    print(f"mean gain from the AR error term, {label:>10}: {by_step.loc[steps, 'gain'].mean():6.1f} pickups per hour")

h = y[:"2023-03-17"].iloc[-168 * 6:]                      # the AR coefficient of one fit, on the last 6 training weeks
phi = SARIMAX(h, exog=slot_dummies(h.index), order=(1, 0, 0), trend="c").fit(disp=False, maxiter=200).params["ar.L1"]
print(f"fitted AR(1) coefficient: {phi:.2f} per hour  ->  after 24 hours: {phi ** 24:.4f} of the initial deviation")
by_step.round(0).astype(int).T.iloc[:, :8]

mean gain from the AR error term,  steps 1-2:   19.4 pickups per hour
mean gain from the AR error term,  steps 3-6:  -10.2 pickups per hour
mean gain from the AR error term, steps 7-24:    0.8 pickups per hour


fitted AR(1) coefficient: 0.77 per hour  ->  after 24 hours: 0.0021 of the initial deviation


step,1,2,3,4,5,6,7,8
6-week average,209,203,135,127,83,52,95,249
slot dummies + AR(1),184,189,149,138,101,49,97,252
gain,24,14,-14,-11,-18,2,-2,-3


The AR error term helps **only in the first two hours** after the origin (about 24 and 14 pickups), where the last observed deviation from the pattern is still fresh. It slightly *hurts* in steps 3 to 5, and from then on it does nothing, because the fitted coefficient of about 0.77 per hour leaves essentially nothing after a day. That is the same short memory that the autocorrelation plot of the remainder showed, and the net effect on a day-ahead forecast is zero: the MAE is 262 with and without it.

<div class="admonition warning">
<p class="admonition-title">Do not oversell a 5% edge</p>
<p>The AR model scores about 5% better than the four-week average on this fortnight. But the six-week average does exactly as well, and the previous page showed that the best number of weeks flips between windows. Whether the classical model is really better needs many test windows, the job of the backtesting page. Do not conclude anything from 14 days.</p>
</div>

## Which one when?

| Situation | Reasonable first choice | Why |
|---|---|---|
| Short season (4, 7, 12), a few hundred points | ETS or SARIMA | few parameters, well understood, fast |
| Long or several seasons (24 and 168) | regression on calendar slots or harmonics + ARMA errors, or a seasonal average | ETS/SARIMA states explode or become unstable |
| No seasonality, noisy level | ETS(A,N,N) or ARIMA | what they were designed for |
| Many related series | a global ML model (next page) | one model learns from all of them |
| Needs an interpretable decomposition | ETS | level, trend and season are explicit |

## Pitfalls at a glance

| Surprise | Cause | Fix |
|---|---|---|
| ETS or ARIMA forecast is a flat line | no seasonal component for a seasonal series | add the season (ETS `seasonal=`, SARIMA `seasonal_order`) or use regression for the season |
| Smoothing weight is exactly 1 or 0 | the model has found nothing to smooth | read `fit.params`; it is information about the data |
| Result changes a lot with history length | adaptive states chase drift | evaluate several training windows; prefer stabler forms |
| `ConvergenceWarning` | optimiser did not finish | count them (as above), simplify the model, scale inputs, do not just silence them |
| Residuals still autocorrelated | model is missing structure | diagnostics guide; the backtest decides if it matters |
| 168 dummies is "too many parameters" | it is 168 coefficients for ~1,000 points | fine for a regression, but expect noise; compare against the plain seasonal average |

## Check yourself

<details class="question">
<summary>1. The level-only exponential smoother chose a weight of exactly 1. What does that mean, and what forecast does it produce?</summary>

The level is updated to equal the newest observation each time, so the forecast for every future step is simply the **last observed value**: the naive forecast. When a series has a strong cycle and the model has no component for it, the optimiser finds that following the latest value is the least bad option.

</details>

<details class="question">
<summary>2. Why can a model with 168 free seasonal coefficients still not beat a plain average of the same weeks?</summary>

Because that regression's coefficients *are* averages of the weeks it was fitted on, up to the weighting that the error model gives. A more flexible model only helps if it captures something the average misses: a trend, short-term momentum, holidays. Here the only extra it has is a momentum term that helps for two hours, so overall there is no gain.

</details>

<details class="question">
<summary>3. A colleague says: ‘I’ll use ETS with a seasonal period of 168 on a year of hourly data.’ Name two practical concerns.</summary>

(1) **Sensitivity and drift**: as the table showed, how much history you give an ETS model changes its quality, and over a year demand levels change. (2) **Cost and fragility**: 168 seasonal states plus smoothing weights are estimated by an optimiser that can fail to converge or settle on poor values; and the model cannot take holidays or weather as inputs. Alternatives: a regression with calendar and holiday features plus a short-memory error model, or a gradient-boosting model on lag features.

</details>

## Exercises

<details class="question">
<summary>On the daily series, let an automatic search pick the SARIMA order: try <code>p, q</code> in 0 to 2, <code>P, Q</code> in 0 to 1 with <code>d=0, D=1, m=7</code>, choose the lowest AIC <em>on the training data only</em>, and backtest it. Does AIC pick the model that forecasts best?</summary>

```python
import itertools
best = min(((SARIMAX(train_d, order=(p, 0, q), seasonal_order=(P, 1, Q, 7)).fit(disp=False).aic, (p, 0, q), (P, 1, Q, 7))
            for p, q, P, Q in itertools.product([0, 1, 2], [0, 1, 2], [0, 1], [0, 1])), key=lambda t: t[0])
print(best)
bt = fc.backtest(daily, arima(best[1], best[2]), origins=origins, horizon=7)
print(fc.score(bt, train_d, m=7))
```

AIC is computed from the *training fit*, so it measures how well a model describes the past, not how well it forecasts. AIC picks SARIMA(1,0,1)(0,1,0,7). Its backtest MAE is about 5,470: a little better than the seasonal naive forecast (5,597), but worse than the hand-picked SARIMA(1,0,1)(0,1,1,7) (5,369), so the lower AIC did not buy a better forecast.

</details>

<details class="question">
<summary>Change the training window of the slot-dummy model from 6 weeks to 4 and to 8. What happens, and what does it tell you?</summary>

```python
for weeks in (4, 6, 8):
    f = regression_arma(slot_dummies, order=(1, 0, 0), weeks=weeks)
    print(weeks, fc.score(fc.backtest(y, lambda h, i: f(h, i), origins=test_origins, horizon=24), train)["MAE"])
```

The error is about 278 with 4 weeks, 262 with 6 and 256 with 8: more history helps this form, the opposite of what happened to ETS with 168 states. Both are small differences next to the window-to-window noise, so treat them as a tendency, not a finding.

</details>

<details class="question">
<summary>Fit the slot-dummy model with an AR(2) error instead of AR(1). Is it worth it?</summary>

```python
f = regression_arma(slot_dummies, order=(2, 0, 0))
print(fc.score(fc.backtest(y, lambda h, i: f(h, i), origins=test_origins, horizon=24), train)["MAE"])
```

The MAE is about 261 against 262: the extra term changes nothing meaningful. Simpler is better when the extra term does not improve out-of-sample error.

</details>

## Recap

- **ETS** and **ARIMA** are fitted by optimisation. Without a seasonal part, neither beats naive on a strongly seasonal series.
- ETS(A,N,A) with a weekly season beat seasonal naive by about 18% on daily totals; seasonal ARIMA only matched it.
- With long seasons (24, 168) put the season in a **regression** and keep ARMA for the errors. Sharp daily shapes need slot dummies; a few harmonics are too smooth.
- A model with a coefficient per slot is essentially a **seasonal average**, which is why it ties the equivalent baseline. The AR error term helps the first two hours and nets out to nothing over a day.
- Diagnostics (residual autocorrelation, convergence counts) tell you what is wrong; the **backtest** tells you whether it matters.

**Next:** *Machine-learning forecasts with lag and calendar features*.